# 🏧 Reporting Parc GAB — 2025 & 2026 (version optimisée gros volumes)
**Objectif** : Analyser les retraits effectués sur **l'ensemble du parc d'automates** pour identifier quelles banques confrères les utilisent le plus, GAB par GAB et au global (nos porteurs vs confrères).

> 💡 **Notre banque = code `20041`** — tout autre code est considéré comme **confrère**

**Périmètre** : tous les `num_automate`, années 2025 + 2026 (colonne `mois_annee` au format `MMAAAA`, ex: `102025` = octobre 2025)

---

## ⚡ Pourquoi cette version est rapide

Avec 70M+ lignes brutes, pandas sature la RAM et fait crasher le kernel — surtout avec des `.apply()` ligne par ligne
et des `groupby` répétés à chaque graphe sur le DataFrame complet.

**Stratégie adoptée ici :**
1. **Agrégation Hive en amont** (recette Dataiku, *avant* ce notebook) : on réduit 70M lignes individuelles à
   un résumé par `(num_automate, mois_annee, banque_attribution)` — typiquement quelques centaines de milliers
   de lignes au lieu de dizaines de millions.
2. **Une seule agrégation pandas supplémentaire** (cellule 2) pour ajouter les colonnes calculées
   (`type_porteur`, `periode_tri`, etc.) — *vectorisée*, pas de `.apply()` ligne par ligne.
3. **Tous les graphes piochent ensuite dans de petits résumés** (quelques dizaines à quelques milliers de lignes),
   jamais dans le DataFrame de base.
4. **Le DataFrame brut original (`df_raw`) est libéré de la mémoire dès qu'il n'est plus utile.**

➡️ Si le kernel meurt encore après ça, le goulot d'étranglement est très probablement le **chargement initial**
(`get_dataframe()`) avant même que le code Python ne s'exécute — dans ce cas il faut absolument agréger en Hive
en amont (voir le fichier `agregation_hive_parc_gab.sql` fourni à côté de ce notebook), et ne plus jamais charger
les 70M lignes brutes dans Dataiku DSS / pandas.

## 0. Import des librairies

In [ ]:
import gc
import dataiku
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Chargement du dataset

⚠️ **Ce notebook attend un dataset déjà agrégé par Hive** (voir `agregation_hive_parc_gab.sql`), avec une ligne
par `(num_automate, mois_annee, banque_attribution)` — PAS le dataset brut de 70M lignes de retraits individuels.

Si vous chargez encore le dataset brut ici, le `get_dataframe()` ci-dessous sera lui-même la cause du crash,
avant même d'arriver au reste du notebook.

In [ ]:
# ⚠️ Remplacez par le nom exact de votre dataset AGRÉGÉ (sortie de la recette Hive) dans Dataiku
dataset = dataiku.Dataset("NOM_DE_VOTRE_DATASET_AGREGE")

# On force les types dès la lecture pour économiser la RAM (évite que pandas devine des types lourds)
dtypes_attendus = {
    'num_automate'           : 'category',
    'mois_annee'              : 'category',
    'banque_attribution'     : 'category',
    'lib_banque_attribution' : 'category',
}

df_raw = dataset.get_dataframe(dtypes=dtypes_attendus, infer_with_pandas=False)

print(f"✅ Dataset chargé : {df_raw.shape[0]:,} lignes, {df_raw.shape[1]} colonnes")
print(f"🏧 Nb automates distincts : {df_raw['num_automate'].nunique()}")
print(f"💾 Mémoire utilisée : {df_raw.memory_usage(deep=True).sum() / 1024**2:.1f} Mo")
df_raw.head()

## 2. Préparation des données — UNE SEULE PASSE, vectorisée

Toutes les colonnes calculées sont ajoutées ici en une seule fois, avec des opérations **vectorisées**
(jamais de `.apply()` ligne par ligne, qui est ~50-100x plus lent qu'une opération vectorisée pandas/numpy
sur de gros volumes).

### 2.1 Typage des colonnes numériques

In [ ]:
df = df_raw.copy()

# --- Typage (vectorisé) ---
df['nb_retraits']    = pd.to_numeric(df['nb_retraits'],    errors='coerce').fillna(0)
df['montant_total'] = pd.to_numeric(df['montant_total'], errors='coerce').fillna(0)

print(f"Lignes : {len(df):,}")
print(f"Valeurs mois_annee distinctes : {df['mois_annee'].nunique()}")

### 2.2 Parsing vectorisé de `mois_annee` (format MMAAAA)

⚠️ Attention : `mois_annee` n'est **pas toujours sur 6 caractères**. Un mois à un chiffre (janvier à septembre)
donne un code à **5 caractères** (ex: `12025` = janvier 2025), alors qu'un mois à deux chiffres
(octobre à décembre) donne **6 caractères** (ex: `102025` = octobre 2025).

**Optimisation clé** : on ne parse pas chaque `mois_annee` ligne par ligne avec `.apply()`. On extrait d'abord
les `mois_annee` **distincts** (il y en a au maximum ~24 sur 2 ans, peu importe le nombre de lignes), on les
parse une seule fois dans une petite table de correspondance, puis on fait un `.map()` vectorisé sur la
colonne complète. Le coût du parsing devient indépendant du nombre de lignes du DataFrame.

In [ ]:
def parse_mois_annee_unique(x):
    """Parse un seul code mois_annee MMAAAA (5 ou 6 caractères) -> dict."""
    x = str(x).strip()
    if len(x) == 6:
        mois, annee = int(x[:2]), int(x[2:])
    elif len(x) == 5:
        mois, annee = int(x[:1]), int(x[1:])
    else:
        return {'annee': None, 'mois': None, 'periode_tri': None, 'periode_label': None}

    periode_tri   = f"{annee:04d}-{mois:02d}"
    mois_labels   = ['', 'Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun',
                      'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']
    periode_label = f"{mois_labels[mois]} {annee}"
    return {'annee': annee, 'mois': mois, 'periode_tri': periode_tri, 'periode_label': periode_label}

# Parsing sur les valeurs DISTINCTES seulement (quelques dizaines max, pas des millions)
mois_annee_uniques = df['mois_annee'].cat.categories if df['mois_annee'].dtype.name == 'category' else df['mois_annee'].unique()
table_correspondance = pd.DataFrame({'mois_annee': mois_annee_uniques})
table_correspondance = table_correspondance.join(
    pd.DataFrame([parse_mois_annee_unique(x) for x in table_correspondance['mois_annee']])
)

# Vérification du parsing
nb_echecs = table_correspondance['periode_tri'].isna().sum()
if nb_echecs > 0:
    print(f"⚠️ {nb_echecs} valeurs mois_annee n'ont pas pu être parsées — à vérifier !")
    display(table_correspondance[table_correspondance['periode_tri'].isna()])
else:
    print(f"✅ {len(table_correspondance)} valeurs mois_annee distinctes, toutes parsées correctement.")

# Jointure vectorisée (map) sur la colonne complète, pas de .apply() ligne par ligne
mapping = table_correspondance.set_index('mois_annee')
df['annee']         = df['mois_annee'].map(mapping['annee']).astype('Int64')
df['mois']          = df['mois_annee'].map(mapping['mois']).astype('Int64')
df['periode_tri']   = df['mois_annee'].map(mapping['periode_tri']).astype('category')
df['periode_label'] = df['mois_annee'].map(mapping['periode_label']).astype('category')

print(f"\nPériode couverte : {df['periode_tri'].min()} → {df['periode_tri'].max()}")
print(f"Années présentes : {sorted(df['annee'].dropna().unique())}")

### 2.3 Colonnes nos porteurs / confrères (vectorisé avec `np.where`)

`np.where` est une comparaison vectorisée en C — bien plus rapide qu'un `.apply()` avec une fonction Python
appelée pour chaque ligne, surtout utile ici si le dataset agrégé reste volumineux.

In [ ]:
NOTRE_BANQUE = '20041'

df['type_porteur'] = np.where(
    df['banque_attribution'].astype(str) == NOTRE_BANQUE,
    'Nos porteurs',
    'Confrères'
)
df['type_porteur'] = df['type_porteur'].astype('category')

# Label affiché : code + libellé (concat vectorisée)
df['label_banque'] = (
    df['banque_attribution'].astype(str) + ' - ' + df['lib_banque_attribution'].astype(str).fillna('Inconnu')
).astype('category')

print(f"Nos porteurs  : {df.loc[df['type_porteur']=='Nos porteurs', 'nb_retraits'].sum():,.0f} retraits")
print(f"Confrères     : {df.loc[df['type_porteur']=='Confrères', 'nb_retraits'].sum():,.0f} retraits")

### 2.4 Libération de la mémoire

On n'a plus besoin de `df_raw` (la copie brute) une fois `df` enrichi — on le supprime explicitement
et on force le garbage collector. Sur de gros volumes, garder deux copies du DataFrame en mémoire en même
temps est souvent ce qui fait déborder la RAM disponible du kernel.

In [ ]:
del df_raw
gc.collect()

print(f"💾 Mémoire de df après optimisation : {df.memory_usage(deep=True).sum() / 1024**2:.1f} Mo")
print(f"Lignes : {len(df):,} | Colonnes : {df.shape[1]}")

## 3. Agrégations de référence — calculées UNE SEULE FOIS

Tous les graphes ci-dessous ne recalculeront plus jamais un `groupby` sur `df` en entier : ils piochent
dans ces quelques résumés, déjà petits (de quelques lignes à quelques milliers de lignes maximum),
calculés une bonne fois pour toutes ici.

In [ ]:
# --- Résumé par type de porteur (2 lignes) ---
agg_type = df.groupby('type_porteur', observed=True).agg(
    nb_retraits    = ('nb_retraits',    'sum'),
    montant_total = ('montant_total', 'sum')
).reset_index()

# --- Résumé par banque (quelques dizaines de lignes max) ---
agg_banque = df.groupby(['label_banque', 'type_porteur'], observed=True).agg(
    nb_retraits    = ('nb_retraits',    'sum'),
    montant_total = ('montant_total', 'sum')
).reset_index()
agg_banque['montant_moyen'] = agg_banque['montant_total'] / agg_banque['nb_retraits']

# --- Résumé par mois (24 lignes max sur 2 ans) ---
agg_mois = df.groupby(['periode_tri', 'periode_label', 'type_porteur'], observed=True).agg(
    nb_retraits    = ('nb_retraits',    'sum'),
    montant_total = ('montant_total', 'sum')
).reset_index().sort_values('periode_tri')
ordre_periodes = agg_mois['periode_label'].drop_duplicates().tolist()

# --- Résumé par GAB (quelques milliers de lignes max = nb de GAB du parc) ---
agg_gab = df.groupby(['num_automate', 'type_porteur'], observed=True).agg(
    nb_retraits=('nb_retraits', 'sum')
).reset_index()
gab_pivot = agg_gab.pivot(index='num_automate', columns='type_porteur', values='nb_retraits').fillna(0)
gab_pivot['total_retraits'] = gab_pivot.get('Nos porteurs', 0) + gab_pivot.get('Confrères', 0)
gab_pivot['pct_confreres']  = (gab_pivot.get('Confrères', 0) / gab_pivot['total_retraits'] * 100).round(1)
gab_pivot = gab_pivot.reset_index()

# --- Résumé annuel (2-3 lignes) ---
agg_annee = df.groupby(['annee', 'type_porteur'], observed=True).agg(
    nb_retraits    = ('nb_retraits',    'sum'),
    montant_total = ('montant_total', 'sum')
).reset_index()
nb_mois_par_an = df.groupby('annee', observed=True)['periode_tri'].nunique()

# --- Résumé saisonnalité mois x année (24 lignes max) ---
agg_saison = df.groupby(['mois', 'annee'], observed=True).agg(
    nb_retraits=('nb_retraits', 'sum')
).reset_index()

print("✅ Toutes les agrégations de référence sont calculées.")
print(f"   agg_type   : {len(agg_type)} lignes")
print(f"   agg_banque : {len(agg_banque)} lignes")
print(f"   agg_mois   : {len(agg_mois)} lignes")
print(f"   agg_gab    : {len(gab_pivot)} lignes (= nb de GAB)")
print(f"   agg_annee  : {len(agg_annee)} lignes")
print(f"   agg_saison : {len(agg_saison)} lignes")

## 4. KPIs — Vue globale parc 2025-2026

In [ ]:
total_retraits = df['nb_retraits'].sum()
total_montant  = df['montant_total'].sum()
nb_banques     = df['banque_attribution'].nunique()
nb_gab         = df['num_automate'].nunique()
pct_nos        = agg_type.loc[agg_type['type_porteur']=='Nos porteurs', 'nb_retraits'].sum() / total_retraits * 100
pct_confreres  = 100 - pct_nos

print("=" * 55)
print(f"  🏧 Nb automates (GAB) : {nb_gab}")
print(f"  📅 Période            : {ordre_periodes[0]} → {ordre_periodes[-1]}")
print(f"  🔢 Total retraits     : {total_retraits:,.0f}")
print(f"  💶 Total montant      : {total_montant:,.0f} €")
print(f"  🏦 Nb banques         : {nb_banques}")
print(f"  ✅ Nos porteurs       : {pct_nos:.1f}%")
print(f"  🤝 Confrères          : {pct_confreres:.1f}%")
print("=" * 55)

## 5. Graphe 1 — Nos porteurs vs Confrères (parc entier)

Vue d'ensemble : qui retire de l'argent sur nos GAB, au global.

In [ ]:
fig1 = px.pie(
    agg_type,
    names  = 'type_porteur',
    values = 'nb_retraits',
    title  = '🏧 Parc GAB — Répartition Nos Porteurs vs Confrères (Nb retraits, 2025-2026)',
    color  = 'type_porteur',
    color_discrete_map = {
        'Nos porteurs' : '#003f88',
        'Confrères'    : '#e85d04'
    },
    hole = 0.4
)
fig1.update_traces(textposition='inside', textinfo='percent+label')
fig1.show()

## 6. Graphe 2 — Top 10 banques confrères (parc entier)

Quelles banques concurrentes utilisent le plus nos automates ?

In [ ]:
top10_banques = agg_banque.sort_values('nb_retraits', ascending=False).head(10)

fig2 = px.bar(
    top10_banques,
    x      = 'nb_retraits',
    y      = 'label_banque',
    color  = 'type_porteur',
    orientation = 'h',
    title  = '🏧 Parc GAB — Top 10 banques par nombre de retraits (2025-2026)',
    labels = {'nb_retraits': 'Nb retraits', 'label_banque': 'Banque'},
    color_discrete_map = {
        'Nos porteurs' : '#003f88',
        'Confrères'    : '#e85d04'
    },
    text   = 'nb_retraits'
)
fig2.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig2.update_layout(yaxis={'categoryorder': 'total ascending'})
fig2.show()

## 7. Graphe 3 — Évolution mensuelle 2025-2026 (Nos porteurs vs Confrères)

Tri chronologique correct grâce à `periode_tri` (sinon "12025" se classerait après "92025" en tri texte).

In [ ]:
fig3 = px.line(
    agg_mois,
    x      = 'periode_label',
    y      = 'nb_retraits',
    color  = 'type_porteur',
    markers= True,
    title  = '🏧 Parc GAB — Évolution mensuelle des retraits (2025-2026)',
    labels = {'nb_retraits': 'Nb retraits', 'periode_label': 'Mois'},
    color_discrete_map = {
        'Nos porteurs' : '#003f88',
        'Confrères'    : '#e85d04'
    },
    category_orders = {'periode_label': ordre_periodes}
)
fig3.update_xaxes(tickangle=-45)
fig3.show()

## 8. Graphe 4 — Top GAB les plus exposés aux confrères 🎯

C'est le cœur du reporting parc : quels automates sont le plus utilisés par les banques concurrentes
(en % et en volume) ? Ce sont les GAB à surveiller / valoriser en priorité.

In [ ]:
# On exclut les GAB avec très peu de volume (peu représentatifs) — seuil ajustable
SEUIL_MIN_RETRAITS = 30
gab_pivot_filtre = gab_pivot[gab_pivot['total_retraits'] >= SEUIL_MIN_RETRAITS]

top_gab_expo_pct = gab_pivot_filtre.sort_values('pct_confreres', ascending=False).head(15)

fig4 = px.bar(
    top_gab_expo_pct,
    x      = 'pct_confreres',
    y      = 'num_automate',
    orientation = 'h',
    title  = f'🎯 Top 15 GAB les plus exposés aux confrères (% retraits confrères, min. {SEUIL_MIN_RETRAITS} retraits)',
    labels = {'pct_confreres': '% retraits confrères', 'num_automate': 'GAB'},
    text   = 'pct_confreres',
    color_discrete_sequence = ['#e85d04']
)
fig4.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig4.update_layout(yaxis={'categoryorder': 'total ascending'})
fig4.show()

print(f"\nℹ️ {len(gab_pivot) - len(gab_pivot_filtre)} GAB exclus du classement (moins de {SEUIL_MIN_RETRAITS} retraits sur la période)")

### 8.1 Même classement, en volume brut de retraits confrères (pas en %)

In [ ]:
top_gab_expo_volume = gab_pivot.sort_values('Confrères', ascending=False).head(15) if 'Confrères' in gab_pivot.columns else pd.DataFrame()

fig4b = px.bar(
    top_gab_expo_volume,
    x      = 'Confrères',
    y      = 'num_automate',
    orientation = 'h',
    title  = '🎯 Top 15 GAB par volume de retraits confrères (nb retraits, 2025-2026)',
    labels = {'Confrères': 'Nb retraits confrères', 'num_automate': 'GAB'},
    text   = 'Confrères',
    color_discrete_sequence = ['#e85d04']
)
fig4b.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig4b.update_layout(yaxis={'categoryorder': 'total ascending'})
fig4b.show()

## 9. Graphe 5 — Montant moyen par retrait par banque (Top 10)

In [ ]:
moy_top10 = agg_banque.sort_values('nb_retraits', ascending=False).head(10)

fig5 = px.bar(
    moy_top10,
    x      = 'label_banque',
    y      = 'montant_moyen',
    color  = 'type_porteur',
    title  = '🏧 Parc GAB — Montant moyen par retrait — Top 10 banques (2025-2026)',
    labels = {'montant_moyen': 'Montant moyen (€)', 'label_banque': 'Banque'},
    color_discrete_map = {
        'Nos porteurs' : '#003f88',
        'Confrères'    : '#e85d04'
    },
    text_auto = '.0f'
)
fig5.update_layout(xaxis_tickangle=-45)
fig5.show()

## 10. Vision annuelle — Comparaison 2025 vs 2026 📆

On change d'angle : au lieu du mois par mois, on regarde **année par année** pour voir si le poids des confrères
progresse ou recule d'une année sur l'autre.

⚠️ Si 2026 n'est pas terminée au moment de l'analyse, la comparaison brute (volumes totaux) favorise
mécaniquement l'année qui a le plus de mois de données. Pense à vérifier le nombre de mois couverts par année
ci-dessous, et privilégie les **%** et les **moyennes mensuelles** plutôt que les volumes bruts si les deux années
ne sont pas comparables en nombre de mois.

In [ ]:
print("Nombre de mois de données disponibles par année :")
print(nb_mois_par_an)

if nb_mois_par_an.nunique() > 1:
    print("\n⚠️ Les années n'ont pas le même nombre de mois — privilégier les comparaisons en % ou en moyenne mensuelle ci-dessous.")
else:
    print("\n✅ Même nombre de mois pour chaque année — comparaison directe possible.")

### 10.1 KPIs annuels comparés

In [ ]:
kpi_annuel_pivot = agg_annee.pivot(index='annee', columns='type_porteur', values='nb_retraits').fillna(0)
kpi_annuel_pivot['total_retraits'] = kpi_annuel_pivot.get('Nos porteurs', 0) + kpi_annuel_pivot.get('Confrères', 0)
kpi_annuel_pivot['pct_confreres']  = (kpi_annuel_pivot.get('Confrères', 0) / kpi_annuel_pivot['total_retraits'] * 100).round(1)
kpi_annuel_pivot['nb_mois']        = nb_mois_par_an
kpi_annuel_pivot['moy_mensuelle_confreres'] = (kpi_annuel_pivot.get('Confrères', 0) / kpi_annuel_pivot['nb_mois']).round(0)

print("=" * 70)
for annee in sorted(kpi_annuel_pivot.index):
    row = kpi_annuel_pivot.loc[annee]
    print(f"  📅 {int(annee)} ({int(row['nb_mois'])} mois de données)")
    print(f"      Total retraits      : {row['total_retraits']:,.0f}")
    print(f"      Confrères           : {row.get('Confrères', 0):,.0f}  ({row['pct_confreres']:.1f}%)")
    print(f"      Moy. mensuelle confrères : {row['moy_mensuelle_confreres']:,.0f} / mois")
    print("-" * 70)

kpi_annuel_pivot

### 10.2 Graphe — Pie charts annuels côte à côte (Nos porteurs vs Confrères)

In [ ]:
annees_disponibles = sorted(agg_annee['annee'].dropna().unique())

fig_annee_pie = make_subplots(
    rows=1, cols=len(annees_disponibles),
    specs=[[{'type': 'domain'}] * len(annees_disponibles)],
    subplot_titles=[f"{int(a)}" for a in annees_disponibles]
)

for i, annee in enumerate(annees_disponibles):
    sous_agg = agg_annee[agg_annee['annee'] == annee]
    fig_annee_pie.add_trace(
        go.Pie(
            labels=sous_agg['type_porteur'],
            values=sous_agg['nb_retraits'],
            name=str(int(annee)),
            marker=dict(colors=[
                '#003f88' if t == 'Nos porteurs' else '#e85d04'
                for t in sous_agg['type_porteur']
            ]),
            hole=0.4,
            textinfo='percent+label'
        ),
        row=1, col=i+1
    )

fig_annee_pie.update_layout(
    title_text='🏧 Parc GAB — Nos Porteurs vs Confrères, par année',
    showlegend=False
)
fig_annee_pie.show()

### 10.3 Graphe — Évolution YoY du % confrères

La courbe la plus parlante du reporting : **le poids des confrères augmente-t-il ou diminue-t-il dans le temps ?**

In [ ]:
# Pivot pour calculer le % confrères par mois directement (vectorisé, pas de .apply() sur groupby)
pivot_mois = agg_mois.pivot_table(
    index=['periode_tri', 'periode_label'],
    columns='type_porteur',
    values='nb_retraits',
    fill_value=0
).reset_index()
pivot_mois['annee'] = pivot_mois['periode_tri'].str[:4]
pivot_mois['total'] = pivot_mois.get('Nos porteurs', 0) + pivot_mois.get('Confrères', 0)
pivot_mois['pct_confreres'] = (pivot_mois.get('Confrères', 0) / pivot_mois['total'] * 100)
pivot_mois = pivot_mois.sort_values('periode_tri')

fig_pct_evol = px.line(
    pivot_mois,
    x      = 'periode_label',
    y      = 'pct_confreres',
    color  = 'annee',
    markers= True,
    title  = '📈 Parc GAB — % de retraits confrères par mois (2025 vs 2026)',
    labels = {'pct_confreres': '% retraits confrères', 'periode_label': 'Mois', 'annee': 'Année'},
    category_orders = {'periode_label': ordre_periodes}
)
fig_pct_evol.update_xaxes(tickangle=-45)
fig_pct_evol.update_yaxes(ticksuffix='%')
fig_pct_evol.show()

### 10.4 Graphe — Comparaison mois-à-mois superposée (saisonnalité)

Même mois (Jan, Fév, ...) sur l'axe X, une courbe par année : permet de voir si la saisonnalité se répète.

In [ ]:
mois_labels_ordre = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun',
                     'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']

agg_saison_fmt = agg_saison.copy()
agg_saison_fmt['mois_nom'] = agg_saison_fmt['mois'].map(lambda m: mois_labels_ordre[int(m)-1])
agg_saison_fmt['annee'] = agg_saison_fmt['annee'].astype(int).astype(str)

fig_saison = px.line(
    agg_saison_fmt,
    x      = 'mois_nom',
    y      = 'nb_retraits',
    color  = 'annee',
    markers= True,
    title  = '🔁 Parc GAB — Saisonnalité : retraits par mois, 2025 vs 2026 superposés',
    labels = {'nb_retraits': 'Nb retraits (tous porteurs)', 'mois_nom': 'Mois', 'annee': 'Année'},
    category_orders = {'mois_nom': mois_labels_ordre}
)
fig_saison.show()

### 10.5 Graphe — Top banques confrères : qui progresse, qui recule (2025 vs 2026) ?

In [ ]:
agg_banque_annee = df[df['type_porteur'] == 'Confrères'].groupby(
    ['label_banque', 'annee'], observed=True
).agg(nb_retraits=('nb_retraits', 'sum')).reset_index()

# On garde le top 8 des banques confrères tous-temps confondus, pour ne pas surcharger le graphe
top8_banques_confreres = (
    agg_banque_annee.groupby('label_banque')['nb_retraits'].sum()
      .sort_values(ascending=False).head(8).index
)
top_banques_annee_filtre = agg_banque_annee[agg_banque_annee['label_banque'].isin(top8_banques_confreres)].copy()
top_banques_annee_filtre['annee'] = top_banques_annee_filtre['annee'].astype(int).astype(str)

fig_banques_yoy = px.bar(
    top_banques_annee_filtre,
    x      = 'nb_retraits',
    y      = 'label_banque',
    color  = 'annee',
    orientation = 'h',
    barmode = 'group',
    title  = '🏦 Top 8 banques confrères — Comparaison 2025 vs 2026',
    labels = {'nb_retraits': 'Nb retraits', 'label_banque': 'Banque', 'annee': 'Année'},
    text   = 'nb_retraits'
)
fig_banques_yoy.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig_banques_yoy.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_banques_yoy.show()

## 11. Tableau récapitulatif final — Détail par GAB x Banque

⚠️ Ce tableau peut être volumineux (1 ligne par GAB x banque distincte). Avec un dataset déjà agrégé en amont,
il reste largement gérable, mais si l'affichage est lent dans Dataiku, limitez-le avec `.head(N)` ou filtrez
sur un sous-ensemble de GAB avant affichage.

In [ ]:
recap_gab = df.groupby(
    ['num_automate', 'banque_attribution', 'lib_banque_attribution', 'type_porteur'], observed=True
).agg(
    nb_retraits    = ('nb_retraits',    'sum'),
    montant_total = ('montant_total', 'sum')
).reset_index()
recap_gab['montant_moyen'] = (recap_gab['montant_total'] / recap_gab['nb_retraits']).round(2)

# Part du retrait confrère/nos porteurs au sein de CHAQUE GAB
recap_gab['part_au_sein_du_gab'] = (
    recap_gab['nb_retraits'] / recap_gab.groupby('num_automate')['nb_retraits'].transform('sum') * 100
).round(2)

recap_gab = recap_gab.sort_values(['num_automate', 'nb_retraits'], ascending=[True, False])

recap_gab.columns = [
    'GAB', 'Code banque', 'Libellé banque', 'Type porteur',
    'Nb retraits', 'Montant total (€)', 'Montant moyen (€)', 'Part au sein du GAB (%)'
]

print(f"\n📋 Tableau récapitulatif détaillé — Parc GAB — 2025-2026 ({len(recap_gab):,} lignes)")
recap_gab.head(200)  # Affichage limité aux 200 premières lignes — retirez .head(200) si besoin du détail complet

### 11.1 Tableau récapitulatif synthétique — Vue par GAB

In [ ]:
recap_synthese = gab_pivot[['num_automate', 'total_retraits', 'pct_confreres']].copy()
if 'Nos porteurs' in gab_pivot.columns:
    recap_synthese['nb_retraits_nos_porteurs'] = gab_pivot['Nos porteurs']
if 'Confrères' in gab_pivot.columns:
    recap_synthese['nb_retraits_confreres'] = gab_pivot['Confrères']

recap_synthese = recap_synthese.sort_values('total_retraits', ascending=False)
recap_synthese.columns = [c.replace('_', ' ').capitalize() for c in recap_synthese.columns]

print(f"\n📋 Synthèse par GAB — {len(recap_synthese)} automates — 2025-2026")
recap_synthese